# Unit 6 — GroupBy & Split-Apply-Combine 🟡 IMPORTANT
**Phase 2 | Project Prometheus**

---

## 🎯 Why This Matters in Quant Finance

Cross-sectional analysis — **ranking stocks within a universe, normalising by sector, comparing factor exposures** — is all GroupBy.

Factor models live here:
- Rank stocks by momentum within each sector
- Compute sector-level Sharpe ratios
- Z-score returns within a peer group to remove sector effects

> 💡 Once you can GroupBy across a 500-stock universe with custom aggregations, you're doing real factor work.


---
## 1️⃣ The Mental Model: Split → Apply → Combine

```
DataFrame
    │
    ├── Split by key (e.g., Sector)
    │       ├── Group: Technology  → [AAPL, MSFT, NVDA rows]
    │       ├── Group: Financials  → [JPM, BAC, GS rows]
    │       └── Group: Energy      → [XOM, CVX rows]
    │
    ├── Apply function to each group independently
    │
    └── Combine results back into one DataFrame
```

### The Three Operations — Critical Distinction

| Method | Returns | Shape | Use Case |
|--------|---------|-------|----------|
| `.agg()` | **One row per group** | Reduced | Summary stats: mean vol, Sharpe per sector |
| `.transform()` | **Same shape as input** | Preserved | Normalise within groups: sector z-scores |
| `.filter()` | **Subset of original rows** | Filtered | Remove groups that fail a threshold |

> ⚠️ `.agg()` vs `.transform()` is a classic confusion. The key: do you want one row per group, or do you want to broadcast the group stat back to every row?


In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf

# Sector ETFs as our universe
tickers = {
    'XLF': 'Financials', 'XLK': 'Technology', 'XLE': 'Energy',
    'XLV': 'Healthcare', 'XLI': 'Industrials', 'XLP': 'Staples',
    'XLY': 'Consumer Disc', 'XLU': 'Utilities'
}

prices = yf.download(list(tickers.keys()), start='2022-01-01', auto_adjust=True)['Close']
returns = prices.pct_change().dropna()
print(f"Shape: {returns.shape}")
print(returns.tail(3))


In [ ]:
# Stack to long format — required for GroupBy across tickers
ret_long = returns.stack().reset_index()
ret_long.columns = ['Date', 'Ticker', 'Return']
ret_long['Sector'] = ret_long['Ticker'].map(tickers)

print(ret_long.head(10))
print(f"\nTotal rows: {len(ret_long)}")


---
## 2️⃣ `.agg()` — Summary Statistics Per Group

In [ ]:
# .agg() — reduces to one row per group
def sharpe(x):
    # Annualised Sharpe Ratio
    return (x.mean() / x.std()) * np.sqrt(252) if x.std() > 0 else np.nan

sector_stats = ret_long.groupby('Sector')['Return'].agg(
    mean_return='mean',
    annualised_vol=lambda x: x.std() * np.sqrt(252),
    sharpe=sharpe,
    count='count'
)

# Annualise mean return
sector_stats['annualised_return'] = sector_stats['mean_return'] * 252

print("Sector Statistics:")
print(sector_stats.round(4).sort_values('sharpe', ascending=False))


---
## 3️⃣ `.transform()` — Broadcast Group Stats Back to Every Row

This is the one that trips people up. Use it when you want to **add a group-level statistic as a new column** — keeping all original rows intact.


In [ ]:
# .transform() — z-score within each Date (cross-sectional normalisation)
# This removes the market's direction and keeps only relative performance
ret_long['zscore'] = ret_long.groupby('Date')['Return'].transform(
    lambda x: (x - x.mean()) / x.std()
)

# Cross-sectional rank each day (1 = best performer that day)
ret_long['daily_rank'] = ret_long.groupby('Date')['Return'].rank(ascending=False)

print(ret_long[['Date', 'Ticker', 'Sector', 'Return', 'zscore', 'daily_rank']].head(16))


In [ ]:
# Verify z-scoring worked: each day's z-scores should have mean ≈ 0, std ≈ 1
daily_zscore_check = ret_long.groupby('Date')['zscore'].agg(['mean', 'std'])
print("Z-score validation (should be ~0 mean, ~1 std):")
print(daily_zscore_check.describe().round(4))


---
## 4️⃣ Grouping Across Multiple Keys

In [ ]:
# Add Year column for multi-key groupby
ret_long['Year'] = ret_long['Date'].dt.year

# Group by Sector AND Year simultaneously → MultiIndex result
annual_sector = ret_long.groupby(['Sector', 'Year'])['Return'].agg(
    total_return=lambda x: (1 + x).prod() - 1,  # compound return
    vol=lambda x: x.std() * np.sqrt(252)
).round(4)

print("Annual returns by sector:")
print(annual_sector)


In [ ]:
# Monthly winners — which sector won each month?
ret_long['YearMonth'] = ret_long['Date'].dt.to_period('M')

monthly_ret = ret_long.groupby(['YearMonth', 'Ticker', 'Sector'])['Return'].sum().reset_index()
monthly_ret.columns = ['Month', 'Ticker', 'Sector', 'Monthly_Return']

# Rank sectors within each month
monthly_ret['monthly_rank'] = monthly_ret.groupby('Month')['Monthly_Return'].rank(ascending=False)

# Which sector finished #1 most often?
winners = monthly_ret[monthly_ret['monthly_rank'] == 1].groupby('Sector').size()
print("\nTimes each sector ranked #1 by month:")
print(winners.sort_values(ascending=False))


---
## ✅ Self-Check Questions

1. What's the difference between `.agg()` and `.transform()`?
   > *`.agg()` collapses groups — one row per group. `.transform()` keeps original shape — broadcasts group stats back to every row.*

2. How do you apply different functions to different columns in one groupby?
   > *Use dict syntax: `.agg({'col1': 'mean', 'col2': 'std'})`*

3. When does groupby produce a MultiIndex?
   > *When grouping by multiple keys — result has one index level per grouping key.*

4. Why does cross-sectional z-scoring use `.transform()` instead of `.agg()`?
   > *Because you want to keep all original rows and add a new column with each row's z-score relative to its group — same shape as input.*

---
## 🧪 Optional Tasks

- Compute a custom Sharpe ratio per sector using `.agg()` with a lambda function.
- Use `.transform()` to z-score returns within each month. Validate: each month's z-scores should sum to ≈ 0.
- Find which sector had the highest Sharpe ratio in 2022 vs 2023. Did it change?
- Try `.filter()`: remove sectors with fewer than 200 trading days of data.
